# Georgia Election Results QC (Cross-Year)

Lean, year-agnostic notebook for visually spot-checking OpenElections Georgia CSVs (`county-level.csv` and `precinct-level.csv`) against the Georgia Secretary of State portal (`https://results.sos.ga.gov`).

1. **Select Year & Election** — auto-discovers CSVs in any year folder (`2000/`–`2026/`) and prints a compact health banner.
2. **Overall Contest Totals** — compare candidate names, offices, districts, parties, county counts, and totals against the SOS summary page.
3. **County-Level Results** — spot-check individual counties or filter to county-vs-precinct diffs.
4. **Precinct-Level Results** — spot-check individual precincts alongside the SOS precinct view.
5. **Ad-Hoc County Drill-Down & SQL Playground** — inspect all precincts in a county or run custom DuckDB SQL.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 1000)

In [ ]:
# Auto-detect repository root (works regardless of Jupyter working directory)
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd().resolve())
con = duckdb.connect()

In [ ]:
def sql(query_str: str, params=None) -> pd.DataFrame:
    """Run a DuckDB SQL query (optionally parameterized) with clean Int64 vote formatting."""
    rel = con.execute(query_str, params) if params is not None else con.execute(query_str)
    df = rel.df()
    for col in df.columns:
        if "pct" in col.lower() or "percent" in col.lower():
            continue
        if pd.api.types.is_float_dtype(df[col]):
            valid = df[col].dropna()
            if len(valid) > 0 and (valid % 1 == 0).all():
                df[col] = df[col].astype("Int64")
        elif col.endswith(("_votes", "_count", "_diff", "votes", "diff", "count", "counties", "precincts")):
            try:
                df[col] = df[col].astype("Int64")
            except Exception:
                pass
    return df

print(f"Repo root: {REPO_ROOT}")

## 1. Select Year & Election
Pick a year folder to list its available election slugs, then load your target election into DuckDB.

In [ ]:
# Choose any year directory (e.g. "2026", "2025", "2024")
ELECTION_YEAR = "2026"

year_dir = REPO_ROOT / ELECTION_YEAR
county_files = sorted(year_dir.glob("*__county-level.csv"), reverse=True)
available_slugs = [f.name.replace("__county-level.csv", "") for f in county_files]

print(f"Available elections in {ELECTION_YEAR}/ ({len(available_slugs)} found):")
for slug in available_slugs:
    has_p = (year_dir / f"{slug}__precinct-level.csv").exists()
    print(f"  - {slug}  [county: OK | precinct: {'OK' if has_p else 'none'}]")

In [ ]:
# Set ELECTION_SLUG to a slug from above, or leave None to auto-select the latest election in ELECTION_YEAR
ELECTION_SLUG = None  # e.g. "20260310__ga__special" or "20251104__ga__municipal__general__special__psc"
# ELECTION_SLUG = '20260310__ga__special'

if ELECTION_SLUG is None:
    if not available_slugs:
        raise FileNotFoundError(f"No *__county-level.csv files found in {year_dir}")
    ELECTION_SLUG = available_slugs[0]

county_csv = year_dir / f"{ELECTION_SLUG}__county-level.csv"
precinct_csv = year_dir / f"{ELECTION_SLUG}__precinct-level.csv"
has_precinct_file = precinct_csv.exists()

county_sql_path = str(county_csv).replace("'", "''")
con.execute(f"CREATE OR REPLACE VIEW county_data AS SELECT * FROM read_csv_auto('{county_sql_path}', all_varchar=false)")
c_cols = {c.lower() for c in con.table("county_data").columns}
has_county_votes = "votes" in c_cols
has_county_mods = {
    "election_day_votes",
    "advanced_votes",
    "absentee_by_mail_votes",
    "provisional_votes",
}.issubset(c_cols)

COUNTY_VOTE_EXPR = (
    "CAST(COALESCE(votes, 0) AS BIGINT)"
    if has_county_votes
    else "CAST(COALESCE(election_day_votes, 0) + COALESCE(advanced_votes, 0) + "
         "COALESCE(absentee_by_mail_votes, 0) + COALESCE(provisional_votes, 0) AS BIGINT)"
)

if has_precinct_file:
    precinct_sql_path = str(precinct_csv).replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW precinct_data AS SELECT * FROM read_csv_auto('{precinct_sql_path}', all_varchar=false)")
    p_cols = {c.lower() for c in con.table("precinct_data").columns}
    has_precinct_votes = "votes" in p_cols
    has_precinct_mods = {
        "election_day_votes",
        "advanced_votes",
        "absentee_by_mail_votes",
        "provisional_votes",
    }.issubset(p_cols)
    PRECINCT_VOTE_EXPR = (
        "CAST(COALESCE(votes, 0) AS BIGINT)"
        if has_precinct_votes
        else "CAST(COALESCE(election_day_votes, 0) + COALESCE(advanced_votes, 0) + "
             "COALESCE(absentee_by_mail_votes, 0) + COALESCE(provisional_votes, 0) AS BIGINT)"
    )
else:
    has_precinct_votes = False
    has_precinct_mods = False
    PRECINCT_VOTE_EXPR = "0"

# --- Compact Health Banner ---
c_rows = int(sql("SELECT COUNT(*) AS n FROM county_data")["n"][0])
c_dups = int(sql(
    "SELECT COUNT(*) AS n FROM (SELECT county, precinct, office, COALESCE(CAST(district AS VARCHAR), ''), party, candidate "
    "FROM county_data GROUP BY 1,2,3,4,5,6 HAVING COUNT(*) > 1)"
)["n"][0])

c_math_bad = 0
suppressed_counties = []
if has_county_mods:
    if has_county_votes:
        c_math_bad = int(sql(
            "SELECT COUNT(*) AS n FROM county_data "
            "WHERE (election_day_votes IS NOT NULL OR advanced_votes IS NOT NULL "
            "   OR absentee_by_mail_votes IS NOT NULL OR provisional_votes IS NOT NULL) "
            "  AND (election_day_votes IS NULL OR advanced_votes IS NULL "
            "   OR absentee_by_mail_votes IS NULL OR provisional_votes IS NULL "
            "   OR votes <> election_day_votes + advanced_votes + absentee_by_mail_votes + provisional_votes)"
        )["n"][0])
    suppressed_counties = sql(
        "SELECT DISTINCT county FROM county_data WHERE election_day_votes IS NULL ORDER BY county"
    )["county"].tolist()

print(f"Loaded:   {ELECTION_YEAR}/{ELECTION_SLUG}")
if has_precinct_file:
    p_rows = int(sql("SELECT COUNT(*) AS n FROM precinct_data")["n"][0])
    p_dups = int(sql(
        "SELECT COUNT(*) AS n FROM (SELECT county, precinct, office, COALESCE(CAST(district AS VARCHAR), ''), party, candidate "
        "FROM precinct_data GROUP BY 1,2,3,4,5,6 HAVING COUNT(*) > 1)"
    )["n"][0])
    p_math_bad = 0
    if has_precinct_mods and has_precinct_votes:
        p_math_bad = int(sql(
            "SELECT COUNT(*) AS n FROM precinct_data "
            "WHERE (election_day_votes IS NOT NULL OR advanced_votes IS NOT NULL "
            "   OR absentee_by_mail_votes IS NOT NULL OR provisional_votes IS NOT NULL) "
            "  AND (election_day_votes IS NULL OR advanced_votes IS NULL "
            "   OR absentee_by_mail_votes IS NULL OR provisional_votes IS NULL "
            "   OR votes <> election_day_votes + advanced_votes + absentee_by_mail_votes + provisional_votes)"
        )["n"][0])
    diff_cnt = int(sql(f"""
        WITH c AS (
            SELECT county, office, COALESCE(CAST(district AS VARCHAR), '') AS district, party, candidate,
                   SUM({COUNTY_VOTE_EXPR}) AS cv
            FROM county_data GROUP BY 1,2,3,4,5
        ), p AS (
            SELECT county, office, COALESCE(CAST(district AS VARCHAR), '') AS district, party, candidate,
                   SUM({PRECINCT_VOTE_EXPR}) AS pv
            FROM precinct_data GROUP BY 1,2,3,4,5
        )
        SELECT COUNT(*) AS n FROM c FULL OUTER JOIN p USING (county, office, district, party, candidate)
        WHERE COALESCE(cv, -1) <> COALESCE(pv, -1)
    """)["n"][0])
    status = "PASS" if (c_dups + p_dups + c_math_bad + p_math_bad) == 0 else "CHECK ISSUES"
    print(f"Rows:     {c_rows:,} county rows | {p_rows:,} precinct rows")
    print(f"Health:   {status} (dups: {c_dups}/{p_dups} | math_bad: {c_math_bad}/{p_math_bad} | county-precinct diffs: {diff_cnt})")
else:
    status = "PASS" if (c_dups + c_math_bad) == 0 else "CHECK ISSUES"
    print(f"Rows:     {c_rows:,} county rows (no precinct CSV)")
    print(f"Health:   {status} (dups: {c_dups} | math_bad: {c_math_bad})")

if suppressed_counties:
    print(f"Suppressed counties (<NA> modalities): {len(suppressed_counties)} -> {suppressed_counties}")
else:
    print("Suppressed counties (<NA> modalities): 0")

## 2. Overall Contest Totals (Compare to GA SOS Summary Page)
Shows `office`, `district`, `party`, `candidate`, participating county count, county total vs. precinct sum, vote share, and vote-method totals. When `suppressed_counties > 0`, `unallocated_votes` shows the exact vote total coming from privacy-suppressed counties whose county-level modalities are blank (`<NA>`).

In [ ]:
mod_select_c = (
    """,
        CAST(SUM(COALESCE(election_day_votes, 0)) AS BIGINT) AS election_day,
        CAST(SUM(COALESCE(advanced_votes, 0)) AS BIGINT) AS advanced,
        CAST(SUM(COALESCE(absentee_by_mail_votes, 0)) AS BIGINT) AS absentee_mail,
        CAST(SUM(COALESCE(provisional_votes, 0)) AS BIGINT) AS provisional,
        CAST(COUNT(*) FILTER (WHERE election_day_votes IS NULL) AS BIGINT) AS suppressed_counties"""
    if has_county_mods else ""
)
mod_select_out = (
    """,
    c.election_day,
    c.advanced,
    c.absentee_mail,
    c.provisional,
    c.suppressed_counties,
    CAST(c.county_votes - (c.election_day + c.advanced + c.absentee_mail + c.provisional) AS BIGINT) AS unallocated_votes"""
    if has_county_mods else ""
)

if has_precinct_file:
    query_overall = f"""
    WITH c_agg AS (
        SELECT
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            CAST(COUNT(DISTINCT county) AS BIGINT) AS counties,
            CAST(SUM({COUNTY_VOTE_EXPR}) AS BIGINT) AS county_votes{mod_select_c}
        FROM county_data
        GROUP BY 1, 2, 3, 4
    ),
    p_agg AS (
        SELECT
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            CAST(SUM({PRECINCT_VOTE_EXPR}) AS BIGINT) AS precinct_votes_sum
        FROM precinct_data
        GROUP BY 1, 2, 3, 4
    )
    SELECT
        COALESCE(c.office, p.office) AS office,
        COALESCE(c.district, p.district) AS district,
        COALESCE(c.party, p.party) AS party,
        COALESCE(c.candidate, p.candidate) AS candidate,
        c.counties,
        c.county_votes,
        p.precinct_votes_sum,
        CAST(COALESCE(c.county_votes, 0) - COALESCE(p.precinct_votes_sum, 0) AS BIGINT) AS diff,
        ROUND(100.0 * c.county_votes / NULLIF(SUM(c.county_votes) OVER (PARTITION BY COALESCE(c.office, p.office), COALESCE(c.district, p.district)), 0), 2) AS pct_total{mod_select_out}
    FROM c_agg c
    FULL OUTER JOIN p_agg p USING (office, district, party, candidate)
    ORDER BY office, TRY_CAST(district AS INTEGER), district, county_votes DESC NULLS LAST, candidate
    """
else:
    query_overall = f"""
    WITH c_agg AS (
        SELECT
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            CAST(COUNT(DISTINCT county) AS BIGINT) AS counties,
            CAST(SUM({COUNTY_VOTE_EXPR}) AS BIGINT) AS county_votes{mod_select_c}
        FROM county_data
        GROUP BY 1, 2, 3, 4
    )
    SELECT
        c.office, c.district, c.party, c.candidate, c.counties, c.county_votes,
        ROUND(100.0 * c.county_votes / NULLIF(SUM(c.county_votes) OVER (PARTITION BY c.office, c.district), 0), 2) AS pct_total{mod_select_out}
    FROM c_agg c
    ORDER BY c.office, TRY_CAST(c.district AS INTEGER), c.district, c.county_votes DESC, c.candidate
    """

sql(query_overall)

## 3. County-Level Results (Spot-Check Counties)
Compare individual county totals and vote-method breakdowns against the SOS county/locality screen. Set `SPOT_CHECK_COUNTY` to inspect one county, or set `ONLY_DIFFS = True` to view only county-vs-precinct residuals.

In [ ]:
# Optional filters:
SPOT_CHECK_COUNTY = 'Gwinnett'  # e.g. "Floyd", "Bibb", "Fulton", or None for all counties
ONLY_DIFFS = False        # Set True to show only rows where county_votes != precinct_votes_sum

c_mod_cols = (
    """,
        CAST(SUM(election_day_votes) AS BIGINT) AS election_day,
        CAST(SUM(advanced_votes) AS BIGINT) AS advanced,
        CAST(SUM(absentee_by_mail_votes) AS BIGINT) AS absentee_mail,
        CAST(SUM(provisional_votes) AS BIGINT) AS provisional"""
    if has_county_mods else ""
)
c_mod_out = ", c.election_day, c.advanced, c.absentee_mail, c.provisional" if has_county_mods else ""

where_parts, params = [], []
if SPOT_CHECK_COUNTY:
    where_parts.append("county = ?")
    params.append(SPOT_CHECK_COUNTY)
if ONLY_DIFFS and has_precinct_file:
    where_parts.append("COALESCE(c.county_votes, -1) <> COALESCE(p.precinct_votes_sum, -1)")
where_sql = ("WHERE " + " AND ".join(where_parts)) if where_parts else ""

if has_precinct_file:
    query_county = f"""
    WITH c_sum AS (
        SELECT
            county,
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            CAST(SUM({COUNTY_VOTE_EXPR}) AS BIGINT) AS county_votes{c_mod_cols}
        FROM county_data
        GROUP BY 1, 2, 3, 4, 5
    ),
    p_sum AS (
        SELECT
            county,
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            CAST(SUM({PRECINCT_VOTE_EXPR}) AS BIGINT) AS precinct_votes_sum,
            CAST(COUNT(DISTINCT precinct) AS BIGINT) AS precinct_count
        FROM precinct_data
        GROUP BY 1, 2, 3, 4, 5
    )
    SELECT
        COALESCE(c.county, p.county) AS county,
        COALESCE(c.office, p.office) AS office,
        COALESCE(c.district, p.district) AS district,
        COALESCE(c.party, p.party) AS party,
        COALESCE(c.candidate, p.candidate) AS candidate,
        c.county_votes,
        p.precinct_votes_sum,
        p.precinct_count,
        CAST(COALESCE(c.county_votes, 0) - COALESCE(p.precinct_votes_sum, 0) AS BIGINT) AS diff{c_mod_out}
    FROM c_sum c
    FULL OUTER JOIN p_sum p USING (county, office, district, party, candidate)
    {where_sql}
    ORDER BY county, office, TRY_CAST(district AS INTEGER), district, county_votes DESC NULLS LAST, candidate
    """
else:
    query_county = f"""
    WITH c_sum AS (
        SELECT
            county,
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            CAST(SUM({COUNTY_VOTE_EXPR}) AS BIGINT) AS county_votes{c_mod_cols}
        FROM county_data
        GROUP BY 1, 2, 3, 4, 5
    )
    SELECT c.county, c.office, c.district, c.party, c.candidate, c.county_votes{c_mod_out}
    FROM c_sum c
    {where_sql}
    ORDER BY c.county, c.office, TRY_CAST(c.district AS INTEGER), c.district, c.county_votes DESC, c.candidate
    """

df_county = sql(query_county, params or None)
print(f"Showing {len(df_county):,} county row(s) (county={SPOT_CHECK_COUNTY}, only_diffs={ONLY_DIFFS}):")
df_county

## 4. Precinct-Level Results (Spot-Check Precincts)
Filter to a specific county and/or precinct (and optional office/district) to spot-check candidate rows and precinct race totals against the SOS precinct screen. Leave `FILTER_COUNTY` / `FILTER_PRECINCT` as `None` to preview the first precinct.

In [ ]:
# Optional filters (all safe for apostrophes in precinct names):
FILTER_COUNTY = None    # e.g. "Floyd", "Richmond", "Fulton"
FILTER_PRECINCT = None  # e.g. "Everett Springs", "504", "01B"
FILTER_OFFICE = None    # e.g. "State Senate", "U.S. House"
FILTER_DISTRICT = None  # e.g. "53", "14"

if not has_precinct_file:
    print("No precinct-level CSV for this election.")
else:
    # Default preview: pick the first (county, precinct) if no filters are set
    if not any([FILTER_COUNTY, FILTER_PRECINCT, FILTER_OFFICE, FILTER_DISTRICT]):
        first_row = sql("SELECT county, precinct FROM precinct_data ORDER BY county, precinct LIMIT 1")
        active_county = first_row["county"][0]
        active_precinct = first_row["precinct"][0]
        print(f"No filters set — previewing first precinct: {active_county} / {active_precinct}")
    else:
        active_county = FILTER_COUNTY
        active_precinct = FILTER_PRECINCT

    where_parts, params = [], []
    if active_county:
        where_parts.append("county = ?")
        params.append(active_county)
    if active_precinct:
        where_parts.append("precinct = ?")
        params.append(active_precinct)
    if FILTER_OFFICE:
        where_parts.append("office = ?")
        params.append(FILTER_OFFICE)
    if FILTER_DISTRICT:
        where_parts.append("COALESCE(CAST(district AS VARCHAR), '') = ?")
        params.append(str(FILTER_DISTRICT))
    where_sql = ("WHERE " + " AND ".join(where_parts)) if where_parts else ""

    p_mod_select = (
        ", election_day_votes, advanced_votes, absentee_by_mail_votes, provisional_votes"
        if has_precinct_mods else ""
    )

    df_prec = sql(f"""
        SELECT
            county,
            precinct,
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            party,
            candidate,
            {PRECINCT_VOTE_EXPR} AS votes{p_mod_select}
        FROM precinct_data
        {where_sql}
        ORDER BY county, precinct, office, TRY_CAST(COALESCE(CAST(district AS VARCHAR), '') AS INTEGER), 4, votes DESC, candidate
    """, params or None)

    df_prec_totals = sql(f"""
        SELECT
            county,
            precinct,
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            CAST(SUM({PRECINCT_VOTE_EXPR}) AS BIGINT) AS race_total_votes,
            CAST(COUNT(DISTINCT candidate) AS BIGINT) AS candidates
        FROM precinct_data
        {where_sql}
        GROUP BY 1, 2, 3, 4
        ORDER BY county, precinct, office, TRY_CAST(COALESCE(CAST(district AS VARCHAR), '') AS INTEGER), 4
    """, params or None)

    print(f"Precinct rows ({len(df_prec):,}):")
    display(df_prec)
    print("Precinct race totals:")
    display(df_prec_totals)

## 5. Ad-Hoc County Drill-Down & SQL Playground
Browse every precinct inside a specific county (with breakout and zero-suppression status), or run custom DuckDB SQL queries directly against `county_data` and `precinct_data`.

In [ ]:
# Pick a county to list all of its precincts and spot suppressed/zero precincts:
DRILL_COUNTY = None    # e.g. "Floyd", "Fulton", "Clarke" (None defaults to first county)
DRILL_OFFICE = None    # Optional, e.g. "State Senate"
DRILL_DISTRICT = None  # Optional, e.g. "53"

if not has_precinct_file:
    print("No precinct-level CSV for this election.")
else:
    target_county = DRILL_COUNTY or sql("SELECT county FROM precinct_data ORDER BY county LIMIT 1")["county"][0]
    where_parts, params = ["county = ?"], [target_county]
    if DRILL_OFFICE:
        where_parts.append("office = ?")
        params.append(DRILL_OFFICE)
    if DRILL_DISTRICT:
        where_parts.append("COALESCE(CAST(district AS VARCHAR), '') = ?")
        params.append(str(DRILL_DISTRICT))
    where_sql = "WHERE " + " AND ".join(where_parts)

    if has_precinct_mods:
        status_expr = """
            CASE
                WHEN SUM({v}) = 0 AND COUNT(election_day_votes) = 0 THEN 'ZERO_SUPPRESSED'
                WHEN SUM({v}) = 0 THEN 'TRUE_ZERO'
                WHEN COUNT(election_day_votes) = 0 THEN 'MODALITIES_SUPPRESSED'
                ELSE 'POPULATED'
            END AS breakout_status
        """.format(v=PRECINCT_VOTE_EXPR)
    else:
        status_expr = f"CASE WHEN SUM({PRECINCT_VOTE_EXPR}) = 0 THEN 'ZERO_VOTES' ELSE 'TOTALS_ONLY' END AS breakout_status"

    df_drill = sql(f"""
        SELECT
            county,
            office,
            COALESCE(CAST(district AS VARCHAR), '') AS district,
            precinct,
            CAST(SUM({PRECINCT_VOTE_EXPR}) AS BIGINT) AS precinct_votes,
            CAST(COUNT(DISTINCT candidate) AS BIGINT) AS candidates,
            {status_expr}
        FROM precinct_data
        {where_sql}
        GROUP BY 1, 2, 3, 4
        ORDER BY office, TRY_CAST(COALESCE(CAST(district AS VARCHAR), '') AS INTEGER), 3, precinct
    """, params)

    print(f"All precincts in {target_county} ({len(df_drill):,} precinct-race rows):")
    display(df_drill)

In [ ]:
# Ad-hoc DuckDB SQL playground (views available: county_data, precinct_data)
sql('''
    SELECT *
    FROM county_data
    LIMIT 10
''')